In [1]:
!pip -q install "transformers<5" torch sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.5 MB/s eta 0:00:00


In [2]:
from transformers import pipeline
import torch
import textwrap
import re

In [3]:
device = 0 if torch.cuda.is_available() else -1

if device == 0:
    print("GPU is available. Summarization will be faster.")
else:
    print("Using CPU. It will work, but it may be slower.")

GPU is available. Summarization will be faster.


In [4]:
model_name = "sshleifer/distilbart-cnn-12-6"

summarizer = pipeline(
    "summarization",
    model=model_name,
    device=device
)

print("Model loaded successfully:", model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Model loaded successfully: sshleifer/distilbart-cnn-12-6


In [5]:
text = """
Object-oriented programming is a programming paradigm based on objects and classes.
A class is a blueprint for creating objects.
An object is an instance of a class.
Encapsulation is used to protect data inside a class.
Inheritance allows one class to reuse properties and methods from another class.
Polymorphism allows objects to behave differently based on their type.
These concepts help developers write reusable and maintainable software.
"""

result = summarizer(
    text,
    max_length=80,
    min_length=25,
    do_sample=False
)

print(result[0]["summary_text"])

 Object-oriented programming is a programming paradigm based on objects and classes . A class is a blueprint for creating objects . Encapsulation is used to protect data inside a class . Polymorphism allows objects to behave differently based on type .


In [6]:
text = """
Object-oriented programming is a programming paradigm based on objects and classes.
A class is a blueprint for creating objects.
An object is an instance of a class.
Encapsulation is used to protect data inside a class and prevent direct access from outside.
Inheritance allows one class to reuse properties and methods from another class.
Polymorphism allows objects to behave differently based on their type.
Abstraction hides unnecessary details and shows only the important features.
These concepts help developers write reusable, secure, organized, and maintainable software.
Object-oriented programming is commonly used in languages such as Java, Python, C++, and C#.
"""

result = summarizer(
    text,
    max_length=100,
    min_length=35,
    do_sample=False
)

print(result[0]["summary_text"])

 Object-oriented programming is a programming paradigm based on objects and classes . It is commonly used in languages such as Java, Python, C++, and C# . It helps developers write reusable, secure, organized, and maintainable software .


In [7]:
def clean_pdf_text(text):
    # Remove extra spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Remove too many new lines
    text = re.sub(r"\n+", "\n", text)

    # Remove common page number patterns
    text = re.sub(r"Page\s+\d+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\b\d+\s*/\s*\d+\b", "", text)

    # Join broken lines into paragraphs
    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if len(line) > 2:
            cleaned_lines.append(line)

    cleaned_text = " ".join(cleaned_lines)

    # Remove extra spaces again
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    return cleaned_text

In [8]:
def split_text_into_chunks(text, max_words=450):
    words = text.split()
    chunks = []

    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])

        if len(chunk.split()) > 40:
            chunks.append(chunk)

    return chunks

In [11]:
def summarize_long_text(text, max_chunks=10):
    cleaned_text = clean_pdf_text(text)

    # Smaller chunks help the model keep more topics
    chunks = split_text_into_chunks(cleaned_text, max_words=100)

    print("Original word count:", len(text.split()))
    print("Cleaned word count:", len(cleaned_text.split()))
    print("Total chunks:", len(chunks))

    if len(chunks) == 0:
        return {
            "chunk_summaries": [],
            "final_summary": "Text is too short or not suitable for summarization."
        }

    chunk_summaries = []

    for index, chunk in enumerate(chunks[:max_chunks], start=1):
        print(f"Summarizing chunk {index}...")

        summary = summarizer(
            chunk,
            max_length=90,
            min_length=25,
            do_sample=False,
            truncation=True
        )[0]["summary_text"]

        chunk_summaries.append(summary)

    # For study notes, do not summarize again too much
    # Because second summarization may remove important points
    detailed_summary = " ".join(chunk_summaries)

    return {
        "chunk_summaries": chunk_summaries,
        "final_summary": detailed_summary
    }

In [12]:
sample_long_text = """
Machine learning is a branch of artificial intelligence that allows systems to learn patterns from data.
Instead of writing fixed rules manually, developers provide data to a machine learning algorithm.
The algorithm studies the data and creates a model that can make predictions or decisions.
For example, in a student performance system, machine learning can analyze attendance, marks, study hours,
missed deadlines, and quiz scores to predict academic risk.

There are different types of machine learning. Supervised learning uses labeled data, where both input and
output are known. Classification is used to predict categories, such as low risk, medium risk, or high risk.
Regression is used to predict continuous values, such as exam marks or study hours.

Unsupervised learning works with data that does not have labels. It is useful for finding hidden patterns
or groups in data. Reinforcement learning is another type where an agent learns by receiving rewards or
penalties from the environment.

Machine learning projects usually include data collection, data cleaning, preprocessing, feature selection,
model training, testing, evaluation, and deployment. Evaluation metrics help developers understand whether
the model is performing well. Accuracy, precision, recall, and F1-score are common metrics for classification.
"""

summary_result = summarize_long_text(sample_long_text)

print("\nFINAL SUMMARY:\n")
print(textwrap.fill(summary_result["final_summary"], width=100))

print("\nCHUNK SUMMARIES:\n")
for i, summary in enumerate(summary_result["chunk_summaries"], start=1):
    print(f"\nChunk {i}:")
    print(textwrap.fill(summary, width=100))

Original word count: 194
Cleaned word count: 194
Total chunks: 2
Summarizing chunk 1...
Summarizing chunk 2...

FINAL SUMMARY:

 Machine learning is a branch of artificial intelligence that allows systems to learn patterns from
data . Developers provide data to a machine learning algorithm . The algorithm studies the data and
creates a model that can make predictions or decisions . Supervised learning uses labeled data,
where both input and output are known .  Machine learning projects usually include data collection,
data cleaning, preprocessing, feature selection, model training, testing, evaluation, and deployment
. Unsupervised learning works with data that does not have labels . Reinforcement learning is
another type where an agent learns by receiving rewards or penalties from the environment .

CHUNK SUMMARIES:


Chunk 1:
 Machine learning is a branch of artificial intelligence that allows systems to learn patterns from
data . Developers provide data to a machine learning algorit

In [13]:
from transformers import pipeline
import torch
import textwrap
import re

device = 0 if torch.cuda.is_available() else -1

model_name = "google/flan-t5-base"

study_summarizer = pipeline(
    "text2text-generation",
    model=model_name,
    device=device
)

print("Study summarization model loaded:", model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Study summarization model loaded: google/flan-t5-base


In [14]:
def clean_pdf_text(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"Page\s+\d+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\b\d+\s*/\s*\d+\b", "", text)

    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        line = line.strip()
        if len(line) > 2:
            cleaned_lines.append(line)

    cleaned_text = " ".join(cleaned_lines)
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    return cleaned_text

In [15]:
def split_text_into_chunks(text, max_words=220):
    words = text.split()
    chunks = []

    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])

        if len(chunk.split()) > 30:
            chunks.append(chunk)

    return chunks

In [16]:
def summarize_chunk_for_study(chunk):
    prompt = f"""
Create a meaningful student-friendly study summary from the text below.

Rules:
- Keep important definitions.
- Keep important types or categories.
- Keep examples.
- Keep steps or processes.
- Keep important technical terms.
- Do not make it too short.
- Write in clear simple English.

Text:
{chunk}

Study Summary:
"""

    result = study_summarizer(
        prompt,
        max_new_tokens=180,
        do_sample=False
    )

    return result[0]["generated_text"]

In [17]:
def generate_meaningful_study_summary(text, max_chunks=6):
    cleaned_text = clean_pdf_text(text)
    chunks = split_text_into_chunks(cleaned_text, max_words=220)

    print("Original word count:", len(text.split()))
    print("Cleaned word count:", len(cleaned_text.split()))
    print("Total chunks:", len(chunks))

    if len(chunks) == 0:
        return "Text is too short or not suitable for summarization."

    chunk_summaries = []

    for index, chunk in enumerate(chunks[:max_chunks], start=1):
        print(f"Summarizing chunk {index}...")
        summary = summarize_chunk_for_study(chunk)
        chunk_summaries.append(summary)

    combined = "\n".join(chunk_summaries)

    final_prompt = f"""
Combine the following chunk summaries into one useful study note.

Rules:
- Keep the meaning correct.
- Keep important concepts.
- Keep important examples.
- Use clear simple English.
- Use this structure:
  1. Main Summary
  2. Key Points
  3. Important Terms

Chunk Summaries:
{combined}

Final Study Note:
"""

    final_result = study_summarizer(
        final_prompt,
        max_new_tokens=300,
        do_sample=False
    )

    return final_result[0]["generated_text"]

In [18]:
sample_long_text = """
Machine learning is a branch of artificial intelligence that allows systems to learn patterns from data.
Instead of writing fixed rules manually, developers provide data to a machine learning algorithm.
The algorithm studies the data and creates a model that can make predictions or decisions.
For example, in a student performance system, machine learning can analyze attendance, marks, study hours,
missed deadlines, and quiz scores to predict academic risk.

There are different types of machine learning. Supervised learning uses labeled data, where both input and
output are known. Classification is used to predict categories, such as low risk, medium risk, or high risk.
Regression is used to predict continuous values, such as exam marks or study hours.

Unsupervised learning works with data that does not have labels. It is useful for finding hidden patterns
or groups in data. Reinforcement learning is another type where an agent learns by receiving rewards or
penalties from the environment.

Machine learning projects usually include data collection, data cleaning, preprocessing, feature selection,
model training, testing, evaluation, and deployment. Evaluation metrics help developers understand whether
the model is performing well. Accuracy, precision, recall, and F1-score are common metrics for classification.
"""

study_note = generate_meaningful_study_summary(sample_long_text)

print("\nFINAL STUDY NOTE:\n")
print(textwrap.fill(study_note, width=100))

Original word count: 194
Cleaned word count: 194
Total chunks: 1
Summarizing chunk 1...

FINAL STUDY NOTE:

This is a good study note for students who want to learn about machine learning and evaluation
metrics.


In [19]:
from transformers import pipeline
import torch
import textwrap
import re

device = 0 if torch.cuda.is_available() else -1

model_name = "facebook/bart-large-cnn"

better_summarizer = pipeline(
    "summarization",
    model=model_name,
    device=device
)

print("Better summarization model loaded:", model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Better summarization model loaded: facebook/bart-large-cnn


In [20]:
def split_text_into_chunks(text, max_words=180):
    words = text.split()
    chunks = []

    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])

        if len(chunk.split()) > 30:
            chunks.append(chunk)

    return chunks

In [24]:
def generate_better_study_summary(text, max_chunks=10):
    cleaned_text = clean_pdf_text(text)

    # Smaller chunks preserve more details
    chunks = split_text_into_chunks(cleaned_text, max_words=70)

    print("Original word count:", len(text.split()))
    print("Cleaned word count:", len(cleaned_text.split()))
    print("Total chunks:", len(chunks))

    if len(chunks) == 0:
        return {
            "summary": "Text is too short or not suitable for summarization.",
            "chunk_summaries": []
        }

    chunk_summaries = []

    for index, chunk in enumerate(chunks[:max_chunks], start=1):
        print(f"Summarizing chunk {index}...")

        summary = better_summarizer(
            chunk,
            max_length=90,
            min_length=25,
            do_sample=False
        )[0]["summary_text"]

        chunk_summaries.append(summary)

    # Do not summarize again, because final summarization removes important details
    final_summary = "\n\n".join(chunk_summaries)

    return {
        "summary": final_summary,
        "chunk_summaries": chunk_summaries
    }

In [25]:
better_result = generate_better_study_summary(sample_long_text, max_chunks=8)

print("\nBETTER STUDY SUMMARY:\n")
print(textwrap.fill(better_result["summary"], width=100))

print("\nCHUNK SUMMARIES:\n")
for i, summary in enumerate(better_result["chunk_summaries"], start=1):
    print(f"\nChunk {i}:")
    print(textwrap.fill(summary, width=100))

Your max_length is set to 90, but your input_length is only 83. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=41)


Original word count: 194
Cleaned word count: 194
Total chunks: 3
Summarizing chunk 1...


Your max_length is set to 90, but your input_length is only 88. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=44)


Summarizing chunk 2...


Your max_length is set to 90, but your input_length is only 74. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=37)


Summarizing chunk 3...

BETTER STUDY SUMMARY:

Machine learning is a branch of artificial intelligence that allows systems to learn patterns from
data. Instead of writing fixed rules manually, developers provide data to a machine learning
algorithm. The algorithm studies the data and creates a model that can make predictions or
decisions.  Supervised learning uses labeled data, where both input and output are known.
Unsupervised learning works with data that does not have labels. Reinforcement learning is useful
for finding hidden patterns or groups in data.  Machine learning projects usually include data
collection, data cleaning, preprocessing, feature selection, model training, testing, evaluation,
and deployment. Evaluation metrics help developers understand whether the model is performing well.

CHUNK SUMMARIES:


Chunk 1:
Machine learning is a branch of artificial intelligence that allows systems to learn patterns from
data. Instead of writing fixed rules manually, developers pro

In [8]:
def generate_better_study_summary(text, max_chunks=10):
    cleaned_text = clean_pdf_text(text)

    # Smaller chunks preserve more details
    chunks = split_text_into_chunks(cleaned_text, max_words=70)

    print("Original word count:", len(text.split()))
    print("Cleaned word count:", len(cleaned_text.split()))
    print("Total chunks:", len(chunks))

    if len(chunks) == 0:
        return {
            "summary": "Text is too short or not suitable for summarization.",
            "chunk_summaries": []
        }

    selected_chunks = chunks[:max_chunks]

    print("Summarizing chunks as batch...")

    summaries = better_summarizer(
        selected_chunks,
        max_length=60,
        min_length=20,
        do_sample=False
    )

    chunk_summaries = [item["summary_text"] for item in summaries]

    final_summary = "\n\n".join(chunk_summaries)

    return {
        "summary": final_summary,
        "chunk_summaries": chunk_summaries
    }

In [9]:
def extract_important_sentences(text, max_sentences=10):
    cleaned_text = clean_pdf_text(text)

    sentences = re.split(r'(?<=[.!?])\s+', cleaned_text)
    sentences = [s.strip() for s in sentences if len(s.split()) > 6]

    if len(sentences) == 0:
        return []

    if len(sentences) <= max_sentences:
        return sentences

    scores = []

    for sentence in sentences:
        words = sentence.split()
        lower = sentence.lower()

        score = 0

        # Longer meaningful sentences usually contain more information
        score += min(len(words), 30)

        # Definition-style sentences are useful for study notes
        if "is a" in lower or "refers to" in lower or "means" in lower:
            score += 8

        # Example sentences are useful
        if "for example" in lower or "such as" in lower:
            score += 10

        # Sentences with lists often contain important concepts
        if "," in sentence:
            score += 5

        scores.append((score, sentence))

    top_sentences = sorted(scores, key=lambda x: x[0], reverse=True)[:max_sentences]
    selected_sentences = [sentence for score, sentence in top_sentences]

    ordered_sentences = [s for s in sentences if s in selected_sentences]

    return ordered_sentences

In [10]:
def generate_studypulse_summary(text, max_chunks=10):
    bart_result = generate_better_study_summary(text, max_chunks=max_chunks)
    important_sentences = extract_important_sentences(text, max_sentences=10)

    return {
        "ai_summary": bart_result["summary"],
        "important_points": important_sentences
    }

In [11]:
sample_long_text = """
Machine learning is a branch of artificial intelligence that allows systems to learn patterns from data.
Instead of writing fixed rules manually, developers provide data to a machine learning algorithm.
The algorithm studies the data and creates a model that can make predictions or decisions.
For example, in a student performance system, machine learning can analyze attendance, marks, study hours,
missed deadlines, and quiz scores to predict academic risk.

There are different types of machine learning. Supervised learning uses labeled data, where both input and
output are known. Classification is used to predict categories, such as low risk, medium risk, or high risk.
Regression is used to predict continuous values, such as exam marks or study hours.

Unsupervised learning works with data that does not have labels. It is useful for finding hidden patterns
or groups in data. Reinforcement learning is another type where an agent learns by receiving rewards or
penalties from the environment.

Machine learning projects usually include data collection, data cleaning, preprocessing, feature selection,
model training, testing, evaluation, and deployment. Evaluation metrics help developers understand whether
the model is performing well. Accuracy, precision, recall, and F1-score are common metrics for classification.
"""

In [7]:
final_result = generate_studypulse_summary(sample_long_text)

print("\nIMPORTANT POINTS:\n")
for i, point in enumerate(final_result["important_points"], start=1):
    print(f"{i}. {point}")

print("\nAI SUMMARY:\n")
print(textwrap.fill(final_result["ai_summary"], width=100))

NameError: name 'generate_better_study_summary' is not defined